# Lab 4: Grammar writing
### COSC 426: Fall 2025, Colgate University

Use this notebook to answer the questions in `Lab4.md`. Make sure to include in this notebook all the tests you run to ensure that at each stage your n-gram model implementation is correct. Please use **markdown** chunks to answer any non-code questions. 

## Part 1

### Part 1.1

(the, panda)-> 1/3;

(a, sandwich)-> 1/2;

(in, the)-> 0;

(panda, eats)-> 1/3;

### Part 1.2

([BOS], the)-> 2/3

([UNK], panda)-> 1/5

(likes, [UNK])-> 1/2

([UNK], peacefully)-> 0

([UNK], [UNK])-> 2/5

(panda, [EOS])-> 1/3

(likes, sandwich)-> 0

Why is it useful to have [BOS] and [EOS] tokens?

- Because it helps us recognize the first and the last words, which makes the calculation of the sentence probability more accurate and prevent codes/models from failing. 

Why is it a useful strategy to replace words not in the vocabulary with the [UNK] token?

- So that we don't assign 0 probabilities to them, because we want to know the context of certain unknown words appear in the bigram model, and it will better inform us how the model perform. 


### Part 1.3

(the, panda)-> 2/14

(a, sandwich)-> 2/13

(in, the)-> 1/11

(panda, eats)-> 2/14

([BOS], the)-> 3/14

([UNK], [UNK])-> 3/16

(panda, [EOS])-> 2/14

What is the motivation for smoothing? If it is easier to articulate your answer, describe a scenario where the absence of smoothing is a problem.

- The motivation would be that we do not fully trust the model we trained on with limited vocabs and three sentences. We want to redistribute the probabilities to count for some of the unseen but definitely possible words. for example, the prob of (in,the) was 0 before smoothing, which is not an ideal situation. But after smoothing, it prevents any 0 probabilities. 

What happens to the probability of observed bigrams with smoothing?

- The larger ones get reduced in general, while the 0 ones bumped up. 

Which of the following smoothing types will be closest to the MLE estimate (i.e., unsmoothed estimate) for observed bigrams? add-1, add-2, add-0.1?

- add-0.1, because the lesser you add, the closer you are to the unsmoothed estimate. 

If the data that you are testing on is very similar to the data used to train your bigram model, would you want to use higher or lower values of k in add-k smoothing? Why?

- We want a lower value of k in add-k because the test data is so similar to the train data that we are very confident in our model and we do not need a large k for probability redistribution. 


## Part 2

Use this part to test your code. Feel free to add any additional code chunks. Note, once you make changes to `Lab4.py` you might have to restart the kernel for the changes to be reflected here!

In [1]:
## DO NOT CHANGE THIS CHUNK
import Lab4
import doctest

import nltk
nltk.download('punkt_tab')

doctest.testmod(Lab4, verbose=True) ## runs the doctest

[nltk_data] Downloading package punkt to /Users/huxiyan/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/huxiyan/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Trying:
    TestBigramFreqs(getBigramFreqs(preprocess('data/test.txt', mark_ends=True), getVocab('data/glove_vocab.txt')))
Expecting:
    {1: 70, 2: 3}
ok
Trying:
    TestBigramFreqs(getBigramFreqs(preprocess('data/test.txt', mark_ends=True), getVocab('data/glove_vocab.txt')), print_non1=True)
Expecting:
    {2: [('kitten', 'had'), ('.', '[EOS]'), ('for', 'the')]}
ok
Trying:
    getBigramProb(('one', 'thing'), 'MLE', bigram_freqs=getBigramFreqs(preprocess('data/test.txt', mark_ends=True),getVocab('data/glove_vocab.txt')), unigram_freqs=getUnigramFreqs(preprocess('data/test.txt', mark_ends=True), getVocab('data/glove_vocab.txt')), vocab_size=len(getVocab('data/glove_vocab.txt')))
Expecting:
    1.0
ok
Trying:
    getBigramProb(('one', 'thing'), 'add-1', bigram_freqs=getBigramFreqs(preprocess('data/test.txt', mark_ends=True),getVocab('data/glove_vocab.txt')), unigram_freqs=getUnigramFreqs(preprocess('data/test.txt', mark_ends=True), getVocab('data/glove_vocab.txt')), vocab_size=len(getVo

TestResults(failed=0, attempted=7)

In [2]:
## Correct probs to test your implementation of getBigramProb()
correct_probs_mle = {
        ('one', 'thing'): 1.0,
        ('kitten', 'had'): 0.6666666666666666,
        ('cat', 'had'): 0.0,
        ('had', 'had'): 0.25,
        ('on', 'the'): 0.0,
        ('held', 'a'): 0.0,
        ('zzzzzzz', 'the'): 0.0
    }

correct_probs_add1 = {
        ('one', 'thing'): 4.999950000499995e-06,
        ('kitten', 'had'): 7.499887501687475e-06,
        ('cat', 'had'): 2.4999750002499977e-06,
        ('had', 'had'): 4.999912501531223e-06,
        ('on', 'the'): 2.499981250140624e-06,
        ('held', 'a'): 2.499981250140624e-06,
        ('zzzzzzz', 'the'): 2.4999562507656116e-06
    }

processed_data = Lab4.preprocess('data/test.txt', mark_ends=True)
vocab = Lab4.getVocab('data/glove_vocab.txt')
unigram_freqs = Lab4.getUnigramFreqs(processed_data, vocab)
bigram_freqs = Lab4.getBigramFreqs(processed_data, vocab)

for bigram, correct_prob in correct_probs_mle.items():
    prob = Lab4.getBigramProb(bigram, 'MLE', bigram_freqs = bigram_freqs, unigram_freqs = unigram_freqs, vocab_size = len(vocab))
    assert abs(prob - correct_prob) < 1e-8, f"MLE: For bigram {bigram}, expected {correct_prob} but got {prob}"

for bigram, correct_prob in correct_probs_add1.items():
    prob = Lab4.getBigramProb(bigram, 'add-1', bigram_freqs = bigram_freqs, unigram_freqs = unigram_freqs, vocab_size = len(vocab))
    assert abs(prob - correct_prob) < 1e-8, f"Add1: For bigram {bigram}, expected {correct_prob} but got {prob}"

## Part 3

Use this part to answer questions in Part 3. Add as many code and markdown chunks as is helpful. **Remember, once you make changes to your `Lab4.py`, you need to restart the kernel and reimport the file**. 

In [3]:
import Lab4
vocab = Lab4.getVocab('data/glove_vocab.txt')
print(f"Vocabulary size: {len(vocab)}")

train_text = Lab4.preprocess('data/through_the_looking_glass.txt', mark_ends=True)
bigram_freqs = Lab4.getBigramFreqs(train_text, vocab)
unigram_freqs = Lab4.getUnigramFreqs(train_text, vocab)

test_files = [
    'data/through_the_looking_glass.txt',
    'data/alice_in_wonderland.txt', 
    'data/sherlock_holmes.txt'
]

results = {}

for test_file in test_files:
    try:
        perplexity = Lab4.evaluateBigramModel(test_file, bigram_freqs, unigram_freqs, vocab, 'add-1')
        results[test_file] = perplexity
    except FileNotFoundError:
        print(f"File {test_file} not found!")
        results[test_file] = None
    except Exception as e:
        print(f"Error evaluating {test_file}: {e}")
        results[test_file] = None
        
for test_file, perplexity in results.items():
    if perplexity:
        filename = test_file.split('/')[-1]
        print(f"  {filename:30}: {perplexity:10.2f}")


Vocabulary size: 400003
  through_the_looking_glass.txt :   24632.50
  alice_in_wonderland.txt       :   34684.53
  sherlock_holmes.txt           :   63904.43


## Observations

1. **Training Text Performance**
   - Lowest perplexity as expected, since the model was trained on this exact text
   - The model has seen these exact bigram patterns during training

2. **Alice in Wonderland Performance**
   - Medium perplexity as both books written by Lewis Carroll, sharing similar vocabulary, writing style, and genre

3. **Sherlock Holmes Performance**
   - Highest perplexity
   - Different author
   - Different genre 
   - Different vocabulary domain

## Part 4 (optional)
Use this part to answer questions in Part 3. Add as many code and markdown chunks as is helpful.

In [6]:
import Lab4
import pandas as pd

vocab = Lab4.getVocab('data/glove_vocab.txt')
k_values = [0.00001, 0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 1.5, 2.0]
mark_ends_options = [True, False]
results = []

for mark_ends in mark_ends_options:
    print(f"\nTesting with mark_ends={mark_ends}")
        
    train_text = Lab4.preprocess('data/through_the_looking_glass.txt', mark_ends=mark_ends)
    bigram_freqs = Lab4.getBigramFreqs(train_text, vocab)
    unigram_freqs = Lab4.getUnigramFreqs(train_text, vocab)
    
    for k in k_values:
        print(f"  Testing k={k}...", end=" ")
        
        try:
            smooth_str = f"add-{k}"
            perplexity = Lab4.evaluateBigramModel(
                'data/through_the_looking_glass.txt', 
                bigram_freqs, 
                unigram_freqs, 
                vocab, 
                smooth_str
            )
            
            results.append({
                'k': k,
                'mark_ends': mark_ends,
                'perplexity': perplexity
            })        
            
            print(f"Perplexity: {perplexity:.2f}")    
        except Exception as e:
            print(f"Error: {e}")

df_results = pd.DataFrame(results)

best_result = df_results.loc[df_results['perplexity'].idxmin()]
print()
print(f"Best k: {best_result['k']}")
print(f"Best mark_ends: {best_result['mark_ends']}")
print(f"Best perplexity: {best_result['perplexity']:.2f}")


Testing with mark_ends=True
  Testing k=1e-05... Perplexity: 9999999999.85
  Testing k=0.0001... Perplexity: 31.51
  Testing k=0.001... Perplexity: 79.85
  Testing k=0.01... Perplexity: 393.79
  Testing k=0.1... Perplexity: 3177.53
  Testing k=0.5... Perplexity: 13844.08
  Testing k=1.0... Perplexity: 24632.50
  Testing k=1.5... Perplexity: 33629.70
  Testing k=2.0... Perplexity: 41401.63

Testing with mark_ends=False
  Testing k=1e-05... Perplexity: 9999999999.85
  Testing k=0.0001... Perplexity: 72.18
  Testing k=0.001... Perplexity: 168.62
  Testing k=0.01... Perplexity: 745.25
  Testing k=0.1... Perplexity: 5229.12
  Testing k=0.5... Perplexity: 20579.84
  Testing k=1.0... Perplexity: 35070.23
  Testing k=1.5... Perplexity: 46704.19
  Testing k=2.0... Perplexity: 56505.00

Best k: 0.0001
Best mark_ends: True
Best perplexity: 31.51


In [11]:
best_k = best_result['k']
best_mark_ends = best_result['mark_ends']
best_smooth_str = f"add-{best_k}"

train_text_best = Lab4.preprocess('data/through_the_looking_glass.txt', mark_ends=best_mark_ends)
bigram_freqs_best = Lab4.getBigramFreqs(train_text_best, vocab)
unigram_freqs_best = Lab4.getUnigramFreqs(train_text_best, vocab)

test_files = [
    'data/through_the_looking_glass.txt',
    'data/alice_in_wonderland.txt',
    'data/sherlock_holmes.txt'
]

test_k_values = [0.00001, 0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 1.5, 2.0]
other_text_results = {}

for test_file in ['data/alice_in_wonderland.txt', 'data/sherlock_holmes.txt']:
    filename = test_file.split('/')[-1]
    print(f"\nTesting different k values on {filename}:")
    other_text_results[filename] = {}
    
    for test_k in test_k_values:
        test_smooth = f"add-{test_k}"
        try:
            perplexity = Lab4.evaluateBigramModel(
                test_file, 
                bigram_freqs_best,  
                unigram_freqs_best, 
                vocab, 
                test_smooth,
                mark_ends=best_mark_ends  
            )
            other_text_results[filename][test_k] = perplexity
            
            marker = " ← OUR BEST" if test_k == best_k else ""
            print(f"  k={test_k:>6}: {perplexity:10.2f}{marker}")
            
        except Exception as e:
            print(f"  k={test_k:>6}: Error - {e}")
            other_text_results[filename][test_k] = None


Testing different k values on alice_in_wonderland.txt:
  k= 1e-05: 9999999999.89
  k=0.0001:     413.97 ← OUR BEST
  k= 0.001:     589.08
  k=  0.01:    1618.87
  k=   0.1:    7301.81
  k=   0.5:   22118.66
  k=   1.0:   34684.53
  k=   1.5:   44498.83
  k=   2.0:   52708.57

Testing different k values on sherlock_holmes.txt:
  k= 1e-05: 9999999999.72
  k=0.0001:    3026.05 ← OUR BEST
  k= 0.001:    3241.67
  k=  0.01:    6396.55
  k=   0.1:   19677.41
  k=   0.5:   45484.75
  k=   1.0:   63904.43
  k=   1.5:   77123.71
  k=   2.0:   87617.58


Although it seems our best k is indeed best on other test sets. However we are not fully convinced because we selected hyperparameters that minimize perplexity on Through the Looking Glass. These hyperparameters may not generalize to other texts/domains.

We need three separate datasets because using the same data for multiple purposes leads to overfitting and poor generalization. Best hyperparameters to the training set were likely not optimal for other sets. This demonstrates data leakage where information from our evaluation metric influenced our model selection. A proper approach requires a training set to learn model parameters; a validation set to select hyperparameters that generalize beyond the training data, and a test set for final, unbiased evaluation.